# 10.3 Active reinforcement learning: practice notebook

Let the agent choose its own actions. Balance exploration and exploitation, learn action values with Q-learning and SARSA, see why they differ on a cliff, and scale up with function approximation.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### The exploration function

In [ ]:
# The exploration function of AIMA section 21.3.1: be optimistic about actions tried fewer than Ne times.
R_PLUS, NE = 2.0, 5

def f(u, n):
    return R_PLUS if n < NE else u

# Two actions in one state: the agent has tried "safe" 20 times and "new" only twice.
estimate = {"safe": 0.6, "new": 0.1}
tries = {"safe": 20, "new": 2}
for a in estimate:
    print(a, " estimate", estimate[a], " tries", tries[a], " value used", f(estimate[a], tries[a]))
print("chosen:", max(estimate, key=lambda a: f(estimate[a], tries[a])))

### Q-learning in a corridor

In [ ]:
# Q-learning in a corridor of 4 squares: 0 1 2 3. Square 3 is the goal (+1). Each other step gives 0.
import random
rng = random.Random(4)
Q = {(s, a): 0.0 for s in range(4) for a in ["L", "R"]}
alpha, gamma, eps = 0.5, 0.9, 0.3

for episode in range(6):
    s = 0
    while s != 3:
        if rng.random() < eps:                       # explore
            a = rng.choice(["L", "R"])
        else:                                        # exploit: best known action (R wins ties)
            a = "R" if Q[(s, "R")] >= Q[(s, "L")] else "L"
        s2 = max(0, s - 1) if a == "L" else s + 1
        r = 1.0 if s2 == 3 else 0.0
        best_next = 0.0 if s2 == 3 else max(Q[(s2, "L")], Q[(s2, "R")])
        Q[(s, a)] = Q[(s, a)] + alpha * (r + gamma * best_next - Q[(s, a)])
        s = s2
print({k: round(v, 3) for k, v in Q.items() if v != 0})
print("greedy policy:", ["R" if Q[(s, "R")] >= Q[(s, "L")] else "L" for s in range(3)])

## Your turn

### Task 1: One Q-learning update

Write `q_update(q, r, best_next, alpha, gamma)` that returns the new Q(s, a). best_next is max over a′ of Q(s′, a′) (0 at the goal).

Hint: Target first, then move a fraction alpha of the way.

In [ ]:
def q_update(q, r, best_next, alpha, gamma):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('round(q_update(0.2, 0.0, 0.8, 0.5, 0.9), 4)', 0.46), ('q_update(0.0, 1.0, 0.0, 0.5, 0.9)', 0.5), ('round(q_update(0.81, 0.0, 0.9, 1.0, 0.9), 4)', 0.81)])

### Task 2: ε-greedy action choice

Write `egreedy(qs, eps, rng)`: qs maps actions to Q values. With probability eps (use `rng.random() < eps`) return `rng.choice(sorted(qs))`, otherwise return the action with the largest value.

Hint: Draw one random number to decide between exploring and exploiting.

In [ ]:
import random

def egreedy(qs, eps, rng):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("egreedy({'L': 0.1, 'R': 0.5}, 0.0, random.Random(1))", 'R'), ("8700 < sum(egreedy({'L': 0.1, 'R': 0.5}, 0.2, rng) == 'R' for rng in [random.Random(3)] for _ in range(10000)) < 9300", True), ("egreedy({'a': -1.0, 'b': -2.0, 'c': -0.5}, 0.0, random.Random(2))", 'c')])

### Task 3: Q-learning in the corridor

Write `learn(episodes, seed)` for the corridor 0 1 2 3 (goal 3 gives +1, other steps 0, Left from 0 stays at 0). Use α = 0.5, γ = 0.9, ε = 0.3, start each episode at 0, and break ties towards "R". Return the Q dictionary keyed by (state, action).

Hint: Follow the loop of the example program, but for many episodes.

In [ ]:
import random

def learn(episodes, seed):
    rng = random.Random(seed)
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("round(learn(300, 1)[(2, 'R')], 2)", 1.0), ("round(learn(300, 1)[(0, 'R')], 2)", 0.81), ("all(learn(300, 5)[(s, 'R')] > learn(300, 5)[(s, 'L')] for s in range(3))", True)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: One Q-learning update

Test 3: the value is already right, so it stays at 0.81.

In [ ]:
def q_update(q, r, best_next, alpha, gamma):
    target = r + gamma * best_next
    return q + alpha * (target - q)

In [ ]:
check([('round(q_update(0.2, 0.0, 0.8, 0.5, 0.9), 4)', 0.46), ('q_update(0.0, 1.0, 0.0, 0.5, 0.9)', 0.5), ('round(q_update(0.81, 0.0, 0.9, 1.0, 0.9), 4)', 0.81)])

### Solution to task 2: ε-greedy action choice

Test 2: with ε = 0.2 and 2 actions, R is chosen 0.8 + 0.1 = 90 percent of the time.

In [ ]:
import random

def egreedy(qs, eps, rng):
    if rng.random() < eps:
        return rng.choice(sorted(qs))
    best = None
    for a in qs:
        if best is None or qs[a] > qs[best]:
            best = a
    return best

In [ ]:
check([("egreedy({'L': 0.1, 'R': 0.5}, 0.0, random.Random(1))", 'R'), ("8700 < sum(egreedy({'L': 0.1, 'R': 0.5}, 0.2, rng) == 'R' for rng in [random.Random(3)] for _ in range(10000)) < 9300", True), ("egreedy({'a': -1.0, 'b': -2.0, 'c': -0.5}, 0.0, random.Random(2))", 'c')])

### Solution to task 3: Q-learning in the corridor

The world is deterministic, so even with a fixed α the Q values of the moves to the right settle at 1, 0.9 and 0.81.

In [ ]:
import random

def learn(episodes, seed):
    rng = random.Random(seed)
    Q = {}
    for s in range(4):
        for a in ["L", "R"]:
            Q[(s, a)] = 0.0
    for e in range(episodes):
        s = 0
        while s != 3:
            if rng.random() < 0.3:
                a = rng.choice(["L", "R"])
            elif Q[(s, "R")] >= Q[(s, "L")]:
                a = "R"
            else:
                a = "L"
            if a == "L":
                s2 = max(0, s - 1)
            else:
                s2 = s + 1
            if s2 == 3:
                r = 1.0
                best = 0.0
            else:
                r = 0.0
                best = max(Q[(s2, "L")], Q[(s2, "R")])
            Q[(s, a)] = Q[(s, a)] + 0.5 * (r + 0.9 * best - Q[(s, a)])
            s = s2
    return Q

In [ ]:
check([("round(learn(300, 1)[(2, 'R')], 2)", 1.0), ("round(learn(300, 1)[(0, 'R')], 2)", 0.81), ("all(learn(300, 5)[(s, 'R')] > learn(300, 5)[(s, 'L')] for s in range(3))", True)])